# 🔭 Phase 2 — Feature Engineer & Signal Analyst
## Role 2: Transit Search & Classical Diagnostics

**Project:** AI-enabled Detection of Exoplanets from Noisy Astronomical Light Curves  
**Team:** 4-person pipeline | Role 2 works on outputs from Role 1 (Data Engineer)

---

### 📋 Phase 2 Objectives (Core Deliverables)

| # | Task |
|---|------|
| 1 | Implement **BLS** (Box Least Squares) and **TLS** (Transit Least Squares) period search |
| 2 | Compute transit shape features: depth, duration, ingress/egress time, impact parameter proxy |
| 3 | Phase-fold light curves & generate folded phase plots as ML training thumbnails |
| 4 | Centroid motion analysis — detect source offset indicating background eclipsing binary |
| 5 | Secondary eclipse / odd-even depth tests, out-of-transit variability checks |
| 6 | Compile **feature vectors** (tabular) for the GBM ensemble model in Role 3 |

---

### 📦 Libraries Used
| Library | Purpose |
|---------|--------|
| `transitleastsquares` | TLS period search (physically-motivated transit model) |
| `astropy` / `astropy.timeseries` | BLS period search, time-series utilities |
| `lightkurve` | Load & interact with TESS/Kepler light curves |
| `numpy` | Numerical array operations |
| `pandas` | Feature table compilation |
| `matplotlib` | Plotting phase-folded light curves & diagnostics |
| `h5py` | Load HDF5 files from Role 1 |
| `scipy` | Signal processing, statistical tests |

---
## ⚙️ Cell 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once in Colab)
!pip install transitleastsquares lightkurve h5py astropy numpy pandas matplotlib scipy -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import warnings
warnings.filterwarnings('ignore')

# Astropy — BLS period search
from astropy.timeseries import BoxLeastSquares
import astropy.units as u

# TLS — Transit Least Squares (physically motivated model)
from transitleastsquares import transitleastsquares, cleaned_array, catalog_info

# Scipy for stats and signal processing
from scipy.stats import median_abs_deviation
from scipy.signal import lombscargle

# Lightkurve for downloading real TESS/Kepler data if needed
import lightkurve as lk

print('✅ All libraries imported successfully!')
print(f'   transitleastsquares version: {__import__("transitleastsquares").__version__}')
print(f'   lightkurve version: {lk.__version__}')

---
## 📂 Cell 2: Load Light Curve Data from Role 1 HDF5 File

**What this does:**  
Role 1 (Data Engineer) delivers cleaned light curve data as `.h5` (HDF5) files.  
Each file contains arrays: `time`, `flux`, and metadata like `star_id`.  
We load this file here to begin our transit search.  

If no HDF5 file is available yet, we provide a **fallback** that downloads a real TESS star directly.

In [ ]:
# ─────────────────────────────────────────────────────────
#  Option A: Load from Role 1's HDF5 file
#  Upload your .h5 file to Colab and set the path below
# ─────────────────────────────────────────────────────────
HDF5_FILE = 'cleaned_light_curve.h5'   # <- change to your actual file name

def load_from_hdf5(file_path):
    """Load cleaned time-series data from Role 1's HDF5 output."""
    with h5py.File(file_path, 'r') as f:
        time  = np.array(f['time'])
        flux  = np.array(f['flux'])
        star_id = f.attrs.get('star_id', 'Unknown_Target')
    valid = ~np.isnan(time) & ~np.isnan(flux)
    time, flux = time[valid], flux[valid]
    # Normalize flux around 1.0
    flux = flux / np.median(flux)
    return time, flux, star_id

# ─────────────────────────────────────────────────────────
#  Option B: Fallback — download a real TESS target
# ─────────────────────────────────────────────────────────
def load_from_tess(tic_id='307210830', sector=1):
    """Download a TESS light curve for testing when HDF5 is not yet available."""
    print(f'🔭 Downloading TESS TIC {tic_id} Sector {sector} from MAST...')
    search = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS', sector=sector)
    lc = search.download().normalize()
    lc = lc.remove_nans().remove_outliers(sigma=5)
    time = lc.time.bkjd if hasattr(lc.time, 'bkjd') else lc.time.value
    flux = lc.flux.value
    return time, flux, f'TIC_{tic_id}'

# ─── Auto-detect which method to use ───
if os.path.exists(HDF5_FILE):
    time, flux, STAR_ID = load_from_hdf5(HDF5_FILE)
    print(f'✅ Loaded HDF5: {HDF5_FILE} | Star: {STAR_ID} | Points: {len(time)}')
else:
    print(f'⚠️  HDF5 file not found. Using TESS fallback data for demo.')
    time, flux, STAR_ID = load_from_tess(tic_id='307210830', sector=1)
    print(f'✅ Loaded TESS data | Star: {STAR_ID} | Points: {len(time)}')

print(f'   Time range: {time.min():.2f} — {time.max():.2f} days')
print(f'   Flux median: {np.median(flux):.4f} | Std: {np.std(flux):.6f}')

---
## 📊 Cell 3: Raw Light Curve Inspection Plot

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(time, flux, 'k.', ms=1.5, alpha=0.6, label='Flux')
ax.set_xlabel('Time (days)', fontsize=12)
ax.set_ylabel('Normalized Flux', fontsize=12)
ax.set_title(f'Raw Light Curve — {STAR_ID}', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('01_raw_light_curve.png', dpi=150)
plt.show()
print('Plot saved as 01_raw_light_curve.png')

---
## 🔍 Cell 4: Algorithm 1 — Box Least Squares (BLS)

**What is BLS?**  
Box Least Squares (BLS) is the **industry-standard period-search algorithm** for finding exoplanet transits.  
It works by fitting a rectangular "box" (sudden top-hat dip) at thousands of trial periods and finding  
which period produces the deepest, most consistent dip in brightness.

**Why BLS first?**  
- Computationally fast for an initial sweep over a wide period range  
- Directly implemented in `astropy.timeseries.BoxLeastSquares`  
- Gives us the "best guess" period, transit duration, and epoch  

**Key output:** BLS Power Spectrum → peaks indicate candidate periods

In [ ]:
print('🔳 Running BLS (Box Least Squares) period search...')

# Define period search grid: 0.5 to 15 days
periods_bls = np.linspace(0.5, 15, 5000) * u.day

# Build BLS model
bls_model = BoxLeastSquares(time * u.day, flux)

# Run the periodogram
bls_result = bls_model.power(
    periods_bls,
    duration=[0.05, 0.1, 0.15, 0.2] * u.day   # trial transit durations
)

# Find best period
best_idx_bls    = np.argmax(bls_result.power)
best_period_bls = bls_result.period[best_idx_bls].value
best_power_bls  = bls_result.power[best_idx_bls]
best_t0_bls     = bls_result.transit_time[best_idx_bls].value
best_dur_bls    = bls_result.duration[best_idx_bls].value
best_depth_bls  = bls_result.depth[best_idx_bls]

print(f'\n📌 BLS Best Results:')
print(f'   Period     : {best_period_bls:.4f} days')
print(f'   Transit T0 : {best_t0_bls:.4f} days')
print(f'   Duration   : {best_dur_bls*24:.2f} hours')
print(f'   Depth      : {best_depth_bls*1e6:.0f} ppm ({best_depth_bls*100:.4f}%)')
print(f'   BLS Power  : {best_power_bls:.4f}')

# Plot BLS periodogram
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(bls_result.period.value, bls_result.power, 'b-', lw=0.8, alpha=0.8)
axes[0].axvline(best_period_bls, color='red', ls='--', lw=1.5, label=f'Best: {best_period_bls:.4f} d')
axes[0].set_xlabel('Period (days)')
axes[0].set_ylabel('BLS Power')
axes[0].set_title('BLS Periodogram', fontweight='bold')
axes[0].legend()

# Phase-fold at BLS period
phase_bls = ((time - best_t0_bls) % best_period_bls) / best_period_bls
phase_bls[phase_bls > 0.5] -= 1.0
sort_idx = np.argsort(phase_bls)

axes[1].plot(phase_bls[sort_idx], flux[sort_idx], 'k.', ms=1.5, alpha=0.5)
axes[1].axvline(0, color='red', ls='--', lw=1.5, alpha=0.7)
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Normalized Flux')
axes[1].set_title(f'BLS Phase-folded @ {best_period_bls:.4f} d', fontweight='bold')
axes[1].set_xlim(-0.5, 0.5)

plt.suptitle(f'BLS Results — {STAR_ID}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('02_bls_periodogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as 02_bls_periodogram.png')

---
## 🔍 Cell 5: Algorithm 2 — Transit Least Squares (TLS)

**What is TLS?**  
Transit Least Squares is an **upgrade over BLS**. Instead of a rectangular box, TLS uses a  
**physically accurate transit model** that accounts for:
- Limb darkening (the star is darker at the edges)
- Gradual ingress and egress (the planet enters/exits gradually)
- The realistic shape of a U-shaped transit

**Why TLS after BLS?**  
- More sensitive — detects small planets BLS misses  
- Provides physically meaningful parameters (impact parameter, stellar density proxy)  
- Produces a Signal Detection Efficiency (SDE) score — a key feature for Role 3's GBM model  

**Key output:** TLS Power Spectrum + SDE + precise transit parameters

In [ ]:
print('🌀 Running TLS (Transit Least Squares) period search...')
print('   (This may take 1-3 minutes depending on data length)')

# Clean data for TLS (remove NaNs and infinite values)
time_clean, flux_clean = cleaned_array(time, flux)

# Initialize and run TLS
tls_model  = transitleastsquares(time_clean, flux_clean)
tls_result = tls_model.power(
    period_min=0.5,          # minimum search period in days
    period_max=15.0,         # maximum search period in days
    oversampling_factor=5,   # period grid resolution
    duration_grid_step=1.05  # duration grid spacing
)

print(f'\n📌 TLS Best Results:')
print(f'   Period              : {tls_result.period:.5f} ± {tls_result.period_uncertainty:.5f} days')
print(f'   Transit T0 (epoch)  : {tls_result.T0:.5f} days')
print(f'   Transit Duration    : {tls_result.duration*24:.2f} hours')
print(f'   Transit Depth       : {tls_result.depth*1e6:.0f} ppm ({tls_result.depth*100:.4f}%)')
print(f'   Rp/Rs (radius ratio): {tls_result.rp_rs:.4f}')
print(f'   SDE (signal strength): {tls_result.SDE:.2f}  [>7 = candidate]')
print(f'   Odd-Even mismatch   : {tls_result.odd_even_mismatch:.3f}')
print(f'   FAP (False Alarm)   : {tls_result.FAP:.6f}')
print(f'   Number of transits  : {tls_result.transit_count}')

# Signal Detection Efficiency > 7 is typically a threshold for a planet candidate
if tls_result.SDE > 7:
    print(f'\n🟢 PLANET CANDIDATE DETECTED! SDE = {tls_result.SDE:.2f} (threshold: 7)')
else:
    print(f'\n🔴 Weak signal. SDE = {tls_result.SDE:.2f} — may be noise or false positive.')

In [ ]:
# Plot TLS power spectrum and phase-folded light curve
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# TLS Power Spectrum
axes[0].axvline(tls_result.period, alpha=0.4, lw=3, color='red')
axes[0].plot(tls_result.periods, tls_result.power, color='black', lw=0.8)
axes[0].set_xlim(np.min(tls_result.periods), np.max(tls_result.periods))
axes[0].set_xlabel('Period (days)')
axes[0].set_ylabel('TLS Power (SDE)')
axes[0].set_title(f'TLS Power Spectrum | Best P={tls_result.period:.4f} d', fontweight='bold')

# Phase-folded with TLS model overlay
axes[1].scatter(tls_result.folded_phase, tls_result.folded_y, color='navy',
                s=2, alpha=0.4, zorder=1, label='Data')
axes[1].plot(tls_result.model_folded_phase, tls_result.model_folded_model,
             color='red', lw=2, zorder=2, label='TLS Model')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Normalized Flux')
axes[1].set_title(f'TLS Phase-folded @ {tls_result.period:.4f} d | SDE={tls_result.SDE:.1f}',
                  fontweight='bold')
axes[1].legend()

plt.suptitle(f'TLS Results — {STAR_ID}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('03_tls_power_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as 03_tls_power_spectrum.png')

---
## 📐 Cell 6: Transit Shape Feature Extraction

**What this does:**  
Once TLS finds the best period, we precisely compute the **physical characteristics** of the transit:  
- **Depth** — how much the star dims (proportional to `(Rp/Rs)²`)  
- **Duration** — total time the planet blocks the star  
- **Ingress/Egress time** — time for planet to fully enter/exit the star's disk  
- **Impact parameter proxy** — how centrally the planet crosses the star (`b = 0` = equatorial, `b = 1` = grazing)  

These are key features for the GBM model in Role 3.

In [ ]:
def extract_transit_shape_features(tls_result):
    """
    Extract precise transit shape features from TLS results.
    Returns a dictionary of physical transit parameters.

    Algorithm:
    - Depth       : fractional flux drop at transit centre
    - Duration    : total transit duration T14 (first to last contact) in hours
    - Ingress/Egress: transit flat-bottom vs total duration ratio
    - Impact parameter proxy: derived from T14 and T23 ratio geometry
    """
    period       = tls_result.period              # days
    depth        = tls_result.depth               # fractional flux drop
    duration     = tls_result.duration            # days (T14 — 1st to 4th contact)
    t0           = tls_result.T0                  # epoch (BJD)
    rp_rs        = tls_result.rp_rs              # planet-to-star radius ratio
    sde          = tls_result.SDE
    transit_count = tls_result.transit_count
    odd_even     = tls_result.odd_even_mismatch
    fap          = tls_result.FAP

    duration_hours = duration * 24.0

    # Transit shape ratio: flat bottom fraction
    # Ingress/egress time approximated from rp/Rs and duration geometry
    # T_ingress ≈ duration * rp_rs / (1 + rp_rs) — simplified
    ingress_time_hours = duration_hours * rp_rs / (1 + rp_rs)
    flat_bottom_frac   = 1 - (2 * rp_rs / (1 + rp_rs))  # T23/T14 proxy
    flat_bottom_frac   = max(0.0, flat_bottom_frac)       # clamp

    # Impact parameter proxy: b ≈ cos(i) * a/Rs
    # From light curve geometry: if T23/T14 → 0, transit is grazing (b → 1)
    impact_param_proxy = np.sqrt(1.0 - flat_bottom_frac) if flat_bottom_frac < 1 else 0.0

    # Transit velocity proxy: depth/duration — how fast the planet crosses
    transit_velocity = depth / max(duration_hours, 1e-6)

    features = {
        'star_id'              : STAR_ID,
        'period_days'          : round(period, 5),
        'epoch_t0'             : round(t0, 5),
        'depth_ppm'            : round(depth * 1e6, 1),
        'depth_fraction'       : round(depth, 6),
        'duration_hours'       : round(duration_hours, 3),
        'ingress_time_hours'   : round(ingress_time_hours, 3),
        'flat_bottom_fraction' : round(flat_bottom_frac, 4),
        'impact_param_proxy'   : round(impact_param_proxy, 4),
        'rp_rs'                : round(rp_rs, 5),
        'transit_velocity'     : round(transit_velocity, 6),
        'sde'                  : round(sde, 3),
        'fap'                  : round(fap, 8),
        'transit_count'        : int(transit_count),
        'odd_even_mismatch'    : round(odd_even, 4),
    }
    return features

transit_features = extract_transit_shape_features(tls_result)

print('📐 Transit Shape Features Extracted:')
print('─' * 45)
for key, val in transit_features.items():
    print(f'  {key:<28}: {val}')

---
## 🌀 Cell 7: Phase-Folding & ML Training Thumbnail

**What is Phase-Folding?**  
We take the long light curve and mathematically "stack" each orbit on top of each other.  
If a planet orbits every 5 days, we cut the data into 5-day chunks and overlay them.  
A real planet transit will appear as a **clean, repeating dip** at phase = 0.  

**Why generate thumbnails?**  
The Role 3 ML team uses these **phase-folded images** to train a 1D-CNN model  
to visually distinguish real planet transits from false positives (eclipsing binaries, noise).

In [ ]:
def phase_fold_and_bin(time, flux, period, t0, n_bins=200):
    """
    Phase-fold a light curve at the given period and epoch,
    then bin into n_bins for a clean, noise-reduced view.

    Algorithm:
    1. Compute phase: phi = ((time - t0) mod period) / period
    2. Center phase around 0 (transit at phase=0)
    3. Bin data into equal-width phase bins and take median per bin
    """
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0

    # Sort by phase
    sort_idx = np.argsort(phase)
    phase_sorted = phase[sort_idx]
    flux_sorted  = flux[sort_idx]

    # Bin into n_bins bins
    bins = np.linspace(-0.5, 0.5, n_bins + 1)
    bin_centers, bin_flux, bin_err = [], [], []
    for i in range(n_bins):
        mask = (phase_sorted >= bins[i]) & (phase_sorted < bins[i+1])
        if mask.sum() > 0:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            bin_flux.append(np.median(flux_sorted[mask]))
            bin_err.append(median_abs_deviation(flux_sorted[mask]) / np.sqrt(mask.sum()))

    return (np.array(phase_sorted), np.array(flux_sorted),
            np.array(bin_centers), np.array(bin_flux), np.array(bin_err))

# Phase-fold using TLS best period and epoch
ph, fl, bc, bf, be = phase_fold_and_bin(
    time_clean, flux_clean,
    tls_result.period, tls_result.T0, n_bins=200
)

# ── Full thumbnail plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: All data points (grey) + binned median (blue)
axes[0].plot(ph, fl, '.', color='lightgrey', ms=2, alpha=0.4, label='Raw points')
axes[0].errorbar(bc, bf, yerr=be, fmt='o', color='steelblue', ms=4,
                 capsize=2, elinewidth=1, label='Binned median')
axes[0].plot(tls_result.model_folded_phase - 0.5, tls_result.model_folded_model,
             'r-', lw=2, label='TLS model', zorder=5)
axes[0].set_xlim(-0.5, 0.5)
axes[0].set_xlabel('Phase', fontsize=12)
axes[0].set_ylabel('Normalized Flux', fontsize=12)
axes[0].set_title(f'Phase-folded Light Curve\nP={tls_result.period:.4f} d | Depth={tls_result.depth*1e6:.0f} ppm',
                  fontweight='bold')
axes[0].legend(fontsize=9)

# Right: Zoomed in on transit (±20% of phase)
zoom = 0.15
mask_zoom = (bc >= -zoom) & (bc <= zoom)
axes[1].errorbar(bc[mask_zoom], bf[mask_zoom], yerr=be[mask_zoom],
                 fmt='o', color='steelblue', ms=5, capsize=3, elinewidth=1.5, label='Binned')
model_mask = (tls_result.model_folded_phase - 0.5 >= -zoom) & (tls_result.model_folded_phase - 0.5 <= zoom)
axes[1].plot(tls_result.model_folded_phase[model_mask] - 0.5,
             tls_result.model_folded_model[model_mask], 'r-', lw=2, label='TLS model')
axes[1].set_xlabel('Phase', fontsize=12)
axes[1].set_ylabel('Normalized Flux', fontsize=12)
axes[1].set_title(f'Zoomed Transit | T_dur={tls_result.duration*24:.2f} h | Rp/Rs={tls_result.rp_rs:.3f}',
                  fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle(f'Phase-folded Light Curve — {STAR_ID} (ML Training Thumbnail)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'04_phase_folded_{STAR_ID}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Thumbnail saved as 04_phase_folded_{STAR_ID}.png')

---
## 🔬 Cell 8: Classical Diagnostics — Odd-Even Depth Test

**What is the Odd-Even Test?**  
A real planet produces transits of **identical depth** every orbit.  
An **Eclipsing Binary (EB)** — two stars orbiting each other — produces alternating deep/shallow dips  
(primary eclipse when the larger star is blocked, secondary when the smaller one is blocked).

**How we detect it:**  
We separate all transits into odd-numbered and even-numbered transits and compare their depths.  
A large difference → likely a false positive (eclipsing binary, not a planet).

In [ ]:
def odd_even_depth_test(time, flux, period, t0, duration):
    """
    Odd-Even Depth Test for Eclipsing Binary rejection.

    Algorithm:
    1. Identify individual transit windows using period and t0
    2. Within each window, measure the minimum flux (transit depth)
    3. Separate transits into odd/even numbered
    4. Compare median depths — large mismatch = EB false positive

    Returns: dict with odd_depth, even_depth, mismatch_sigma, is_suspicious
    """
    half_dur = duration / 2.0

    # Predict all transit times
    n_start = int(np.floor((time.min() - t0) / period))
    n_end   = int(np.ceil((time.max() - t0) / period))
    transit_depths = []

    for n in range(n_start, n_end + 1):
        t_center = t0 + n * period
        mask = (time >= t_center - half_dur * 1.5) & (time <= t_center + half_dur * 1.5)
        if mask.sum() > 3:
            depth_n = 1.0 - np.min(flux[mask])
            transit_depths.append((n, depth_n))

    if len(transit_depths) < 2:
        return {'odd_depth': np.nan, 'even_depth': np.nan,
                'mismatch_sigma': 0.0, 'is_suspicious': False}

    odd_depths  = [d for (n, d) in transit_depths if n % 2 != 0]
    even_depths = [d for (n, d) in transit_depths if n % 2 == 0]

    odd_med  = np.median(odd_depths)  if odd_depths  else np.nan
    even_med = np.median(even_depths) if even_depths else np.nan

    # Compute mismatch in units of combined scatter
    combined_err = np.std(odd_depths + even_depths) + 1e-9
    mismatch_sigma = abs(odd_med - even_med) / combined_err

    is_suspicious = mismatch_sigma > 3.0   # >3 sigma = likely EB

    return {
        'odd_depth_ppm'   : round(odd_med * 1e6, 1),
        'even_depth_ppm'  : round(even_med * 1e6, 1),
        'mismatch_sigma'  : round(mismatch_sigma, 3),
        'is_suspicious_eb': is_suspicious,
        'n_transits_found': len(transit_depths)
    }

oe_result = odd_even_depth_test(
    time_clean, flux_clean,
    tls_result.period, tls_result.T0, tls_result.duration
)

print('🔬 Odd-Even Depth Test Results:')
print('─' * 40)
for k, v in oe_result.items():
    print(f'   {k:<28}: {v}')

# Also show TLS's own odd-even metric
print(f'\n   TLS odd_even_mismatch     : {tls_result.odd_even_mismatch:.4f}')
if tls_result.odd_even_mismatch > 3 or oe_result['is_suspicious_eb']:
    print('\n⚠️  WARNING: Odd-even mismatch is high — possible Eclipsing Binary!')
else:
    print('\n✅ Odd-even depths are consistent — no strong EB signature.')

---
## 🌟 Cell 9: Secondary Eclipse Detection

**What is a Secondary Eclipse?**  
When a binary star system eclipses, we see **two dips** per full orbit:  
1. Primary eclipse (at phase = 0) — deeper  
2. Secondary eclipse (at phase = 0.5) — shallower

A real planet emits very little light, so there is **no secondary eclipse** at half-phase.  
Detecting a significant secondary eclipse is strong evidence of an eclipsing binary **false positive**.

In [ ]:
def detect_secondary_eclipse(phase, flux, primary_depth, window=0.05):
    """
    Secondary Eclipse Detection.

    Algorithm:
    1. Phase-fold the light curve (done in Cell 7)
    2. Look at data in a window around phase = 0.5 (half-orbit)
    3. Compute the mean flux dip relative to out-of-transit level
    4. Compare secondary depth to primary depth
    5. Ratio > 0.1 (10%) is suspicious for an EB

    Returns: dict with secondary_depth_ppm, secondary_ratio, is_suspicious
    """
    # Out-of-transit baseline (avoid primary ±0.1 and secondary ±window)
    oot_mask = (np.abs(phase) > 0.1) & (np.abs(np.abs(phase) - 0.5) > window)
    baseline  = np.median(flux[oot_mask]) if oot_mask.sum() > 10 else 1.0

    # Secondary eclipse window: phase = ±window around 0.5
    sec_mask_pos = (phase >= 0.5 - window) & (phase <= 0.5 + window)
    sec_mask_neg = (phase >= -0.5 - window) & (phase <= -0.5 + window)
    sec_mask = sec_mask_pos | sec_mask_neg

    if sec_mask.sum() < 3:
        return {'secondary_depth_ppm': 0.0, 'secondary_ratio': 0.0,
                'is_suspicious_secondary': False}

    sec_flux_min = np.min(flux[sec_mask])
    secondary_depth = max(0.0, baseline - sec_flux_min)
    ratio = secondary_depth / max(primary_depth, 1e-9)

    return {
        'secondary_depth_ppm'    : round(secondary_depth * 1e6, 1),
        'secondary_ratio'        : round(ratio, 4),
        'is_suspicious_secondary': ratio > 0.10
    }

sec_result = detect_secondary_eclipse(ph, fl, primary_depth=tls_result.depth)

print('🌟 Secondary Eclipse Test Results:')
print('─' * 40)
for k, v in sec_result.items():
    print(f'   {k:<30}: {v}')

if sec_result['is_suspicious_secondary']:
    print('\n⚠️  WARNING: Secondary eclipse detected — likely Eclipsing Binary!')
else:
    print('\n✅ No significant secondary eclipse — consistent with a planet.')

# Plot the phase-folded curve highlighting secondary eclipse region
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ph, fl, '.', color='lightgrey', ms=1.5, alpha=0.4)
ax.errorbar(bc, bf, yerr=be, fmt='o', color='steelblue', ms=4, capsize=2, label='Binned data')
ax.axvspan(-0.05, 0.05, alpha=0.2, color='red', label='Primary transit')
ax.axvspan(0.45, 0.55, alpha=0.2, color='orange', label='Secondary eclipse window')
ax.set_xlim(-0.5, 0.5)
ax.set_xlabel('Phase', fontsize=12)
ax.set_ylabel('Normalized Flux', fontsize=12)
ax.set_title(f'Secondary Eclipse Check — {STAR_ID}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('05_secondary_eclipse_check.png', dpi=150)
plt.show()
print('Plot saved as 05_secondary_eclipse_check.png')

---
## 📡 Cell 10: Centroid Motion Analysis

**What is Centroid Motion?**  
If the light dip is caused by a **background eclipsing binary** (a faint binary star near our target),  
then during the dip, the **apparent position (centroid) of the star shifts** toward the binary's location.

**How we detect it:**  
We look at how much the photometric centroid (centre of brightness) moves *in-transit* vs *out-of-transit*.  
A large centroid shift is a strong indicator that the transit signal is **not from the target star** itself.

> **Note:** Full centroid motion requires pixel-level data from TESS (TPF files).  
> Here we compute a **flux-centroid proxy** from the 1D light curve statistics,  
> and flag if the out-of-transit variability changes correlated with transit times.

In [ ]:
def centroid_motion_proxy(time, flux, period, t0, duration):
    """
    Centroid Motion Proxy — detects background eclipsing binary contamination.

    Algorithm (1D proxy since pixel data is not always available):
    1. Separate flux into in-transit and out-of-transit segments
    2. Compute RMS and skewness of each segment
    3. A real planet: OOT flux is flat, in-transit flux shows clean dip
    4. Background EB: OOT flux shows correlated noise from the contaminator
    5. Compute flux centroid shift proxy: |mean(in_transit) - mean(oot)| normalized by oot_std

    For full centroid motion, use lightkurve's TargetPixelFile.centroid_col/row.

    Returns: dict with proxy metrics and contamination flag
    """
    half_dur = duration / 2.0

    # Build in-transit and out-of-transit masks
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    phase_dur_frac = (half_dur * 1.5) / period

    in_transit  = np.abs(phase) < phase_dur_frac
    out_transit = ~in_transit

    if in_transit.sum() < 3 or out_transit.sum() < 10:
        return {'centroid_proxy': 0.0, 'oot_rms_ppm': 0.0,
                'in_transit_mean': 0.0, 'contamination_flag': False}

    oot_flux   = flux[out_transit]
    in_flux    = flux[in_transit]

    oot_mean   = np.median(oot_flux)
    oot_rms    = np.std(oot_flux)
    in_mean    = np.mean(in_flux)

    # Centroid proxy: normalized flux shift during transit
    centroid_proxy = abs(in_mean - oot_mean) / max(oot_rms, 1e-9)

    # OOT variability check: excess variability outside transit?
    oot_variability_ppm = oot_rms * 1e6

    # Contamination flag: if the dip is smaller than 3x OOT noise, suspicious
    expected_snr = abs(in_mean - oot_mean) / max(oot_rms / np.sqrt(in_transit.sum()), 1e-9)
    contamination_flag = expected_snr < 3.0 and centroid_proxy > 1.0

    return {
        'centroid_proxy'       : round(centroid_proxy, 4),
        'oot_rms_ppm'          : round(oot_variability_ppm, 1),
        'in_transit_mean'      : round(in_mean, 6),
        'transit_snr_proxy'    : round(expected_snr, 2),
        'contamination_flag'   : contamination_flag
    }

centroid_result = centroid_motion_proxy(
    time_clean, flux_clean,
    tls_result.period, tls_result.T0, tls_result.duration
)

print('📡 Centroid Motion Analysis Results:')
print('─' * 42)
for k, v in centroid_result.items():
    print(f'   {k:<30}: {v}')

if centroid_result['contamination_flag']:
    print('\n⚠️  WARNING: Possible background contamination — centroid motion detected!')
else:
    print('\n✅ No significant centroid offset — source is likely the target star.')

---
## 📊 Cell 11: Out-of-Transit Variability Check

**What this does:**  
A planet transit produces a **very smooth, stable** out-of-transit (OOT) baseline.  
Stellar variability (starspots, pulsations) or a background star can create  
correlated noise that **mimics** a transit. We quantify the OOT noise level  
and check whether it is consistent with the expected photon noise floor.

In [ ]:
def out_of_transit_variability(time, flux, period, t0, duration):
    """
    Out-of-Transit (OOT) Variability Assessment.

    Algorithm:
    1. Mask in-transit data
    2. Compute CDPP (Combined Differential Photometric Precision) — noise per transit duration
    3. Measure scatter using MAD (robust std), skewness, and kurtosis
    4. Check for periodic signals in OOT using Lomb-Scargle periodogram
       (stellar rotation or binary signals would appear here)

    Returns dict of OOT noise metrics
    """
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    phase_dur_frac = (duration * 1.5) / period
    oot_mask = np.abs(phase) > phase_dur_frac

    oot_time = time[oot_mask]
    oot_flux = flux[oot_mask]

    oot_mad    = median_abs_deviation(oot_flux)
    oot_std    = np.std(oot_flux)
    oot_mean   = np.mean(oot_flux)
    oot_skew   = float(pd.Series(oot_flux).skew())
    oot_kurt   = float(pd.Series(oot_flux).kurtosis())

    # CDPP estimate: noise in transit-duration bins
    n_pts_in_transit = max(1, int(duration * len(time) / (time.max() - time.min())))
    cdpp_estimate = oot_std / np.sqrt(n_pts_in_transit) * 1e6  # in ppm

    # Lomb-Scargle on OOT to detect stellar rotation / binary signals
    if len(oot_time) > 20:
        freqs = np.linspace(1.0 / 30.0, 2.0, 2000)   # 0.5 to 30 day periods
        pgram = lombscargle(oot_time, oot_flux - oot_mean,
                            2 * np.pi * freqs, normalize=True)
        ls_peak_period = 1.0 / freqs[np.argmax(pgram)]
        ls_peak_power  = float(np.max(pgram))
    else:
        ls_peak_period, ls_peak_power = np.nan, np.nan

    return {
        'oot_std_ppm'          : round(oot_std * 1e6, 1),
        'oot_mad_ppm'          : round(oot_mad * 1e6, 1),
        'cdpp_ppm'             : round(cdpp_estimate, 1),
        'oot_skewness'         : round(oot_skew, 4),
        'oot_kurtosis'         : round(oot_kurt, 4),
        'ls_peak_period_days'  : round(ls_peak_period, 3) if not np.isnan(ls_peak_period) else None,
        'ls_peak_power'        : round(ls_peak_power, 4) if not np.isnan(ls_peak_power) else None,
        'high_variability_flag': oot_std * 1e6 > 5000    # >5000 ppm is very noisy
    }

oot_result = out_of_transit_variability(
    time_clean, flux_clean,
    tls_result.period, tls_result.T0, tls_result.duration
)

print('📊 Out-of-Transit Variability Results:')
print('─' * 42)
for k, v in oot_result.items():
    print(f'   {k:<30}: {v}')

if oot_result['high_variability_flag']:
    print('\n⚠️  WARNING: Very high OOT variability — stellar activity or contamination!')
else:
    print('\n✅ OOT variability is within acceptable limits.')

---
## 📋 Cell 12: Compile Full Feature Vector for Role 3 GBM Model

**What this does:**  
All the features computed above are compiled into a **single tabular row** per star.  
This tabular feature vector is exactly what the **Role 3 ML modeller** uses as input  
to train an XGBoost / LightGBM ensemble that classifies:  
- ✅ Real planet (PC — Planet Candidate)  
- ❌ False positive (EB, background binary, noise)

The final CSV file is your **primary deliverable** — hand this to Role 3.

In [ ]:
def compile_feature_vector(
        transit_features, oe_result, sec_result,
        centroid_result, oot_result, tls_result
):
    """
    Compile all diagnostic features into a single-row feature vector
    for the GBM ensemble model in Role 3.

    Categories:
    - Period / Epoch features (BLS & TLS)
    - Transit shape features (depth, duration, Rp/Rs, impact param)
    - Signal quality features (SDE, FAP, transit count)
    - Vetting / diagnostic features (odd-even, secondary, centroid, OOT)
    """
    fv = {}

    # ── BLS features ──
    fv['bls_period_days']       = best_period_bls
    fv['bls_power']             = round(best_power_bls, 5)
    fv['bls_depth_ppm']         = round(best_depth_bls * 1e6, 1)
    fv['bls_duration_hours']    = round(best_dur_bls * 24, 3)

    # ── TLS / Transit shape features ──
    fv.update(transit_features)

    # ── Odd-even test features ──
    fv['oe_odd_depth_ppm']      = oe_result.get('odd_depth_ppm', np.nan)
    fv['oe_even_depth_ppm']     = oe_result.get('even_depth_ppm', np.nan)
    fv['oe_mismatch_sigma']     = oe_result.get('mismatch_sigma', np.nan)
    fv['oe_tls_mismatch']       = round(tls_result.odd_even_mismatch, 4)
    fv['oe_suspicious_flag']    = int(oe_result.get('is_suspicious_eb', False))

    # ── Secondary eclipse features ──
    fv['sec_depth_ppm']         = sec_result.get('secondary_depth_ppm', 0.0)
    fv['sec_ratio']             = sec_result.get('secondary_ratio', 0.0)
    fv['sec_suspicious_flag']   = int(sec_result.get('is_suspicious_secondary', False))

    # ── Centroid / contamination features ──
    fv['centroid_proxy']        = centroid_result.get('centroid_proxy', 0.0)
    fv['centroid_transit_snr']  = centroid_result.get('transit_snr_proxy', 0.0)
    fv['contamination_flag']    = int(centroid_result.get('contamination_flag', False))

    # ── OOT variability features ──
    fv['oot_std_ppm']           = oot_result.get('oot_std_ppm', np.nan)
    fv['oot_cdpp_ppm']          = oot_result.get('cdpp_ppm', np.nan)
    fv['oot_skewness']          = oot_result.get('oot_skewness', np.nan)
    fv['oot_kurtosis']          = oot_result.get('oot_kurtosis', np.nan)
    fv['oot_ls_period']         = oot_result.get('ls_peak_period_days', np.nan)
    fv['oot_ls_power']          = oot_result.get('ls_peak_power', np.nan)
    fv['high_variability_flag'] = int(oot_result.get('high_variability_flag', False))

    # ── Composite vetting score (simple sum of flag features) ──
    n_false_positive_flags = (
        fv['oe_suspicious_flag'] +
        fv['sec_suspicious_flag'] +
        fv['contamination_flag'] +
        fv['high_variability_flag']
    )
    fv['n_fp_flags']            = n_false_positive_flags
    fv['is_planet_candidate']   = int(fv['sde'] > 7 and n_false_positive_flags == 0)

    return fv

feature_vector = compile_feature_vector(
    transit_features, oe_result, sec_result,
    centroid_result, oot_result, tls_result
)

# Print the full feature vector
print('📋 Full Feature Vector for Role 3 GBM Model:')
print('═' * 55)
df_fv = pd.DataFrame([feature_vector])
for col in df_fv.columns:
    print(f'   {col:<32}: {df_fv[col].values[0]}')

print(f'\n   Total features compiled: {len(feature_vector)}')
status = '🟢 PLANET CANDIDATE' if feature_vector['is_planet_candidate'] else '🔴 False Positive'
print(f'   Vetting Result        : {status}')

---
## 💾 Cell 13: Export Feature Table to CSV (Handoff to Role 3)

In [ ]:
OUTPUT_CSV = f'role2_feature_table_{STAR_ID}.csv'

df_export = pd.DataFrame([feature_vector])
df_export.to_csv(OUTPUT_CSV, index=False)

print(f'✅ Feature vector exported to: {OUTPUT_CSV}')
print(f'   Shape: {df_export.shape[0]} row(s) × {df_export.shape[1]} features')
print(f'\nHandoff instructions for Role 3:')
print('  1. Share this CSV file with the ML modeller (Role 3)')
print('  2. Share phase-folded PNG files as CNN training thumbnails')
print('  3. The column "is_planet_candidate" = 1 means planet, 0 = false positive')

display(df_export.T.rename(columns={0: 'Value'}))

---
## 🗂️ Cell 14: Batch Processing Multiple Stars

**Scale up:** Run the full Phase 2 pipeline on a list of stars automatically.

This is what you'll do in practice — Role 1 gives you many HDF5 files, and you process them all.

In [ ]:
def run_phase2_pipeline(time, flux, star_id):
    """
    Full Phase 2 pipeline for one star.
    Returns a feature vector dictionary.
    """
    try:
        # Step 1: BLS
        periods_bls = np.linspace(0.5, 15, 3000) * u.day
        bls_m = BoxLeastSquares(time * u.day, flux)
        bls_r = bls_m.power(periods_bls, duration=[0.05, 0.1, 0.2] * u.day)
        bi    = np.argmax(bls_r.power)
        bp    = bls_r.period[bi].value
        bd    = bls_r.depth[bi]
        bpow  = bls_r.power[bi]
        bdur  = bls_r.duration[bi].value

        # Step 2: TLS
        tc, fc = cleaned_array(time, flux)
        tls_m  = transitleastsquares(tc, fc)
        tls_r  = tls_m.power(period_min=0.5, period_max=15.0,
                             oversampling_factor=3, duration_grid_step=1.05)

        # Step 3: Extract features
        STAR_ID = star_id
        tf  = extract_transit_shape_features(tls_r)

        ph2, fl2, bc2, bf2, be2 = phase_fold_and_bin(tc, fc, tls_r.period, tls_r.T0)
        oe  = odd_even_depth_test(tc, fc, tls_r.period, tls_r.T0, tls_r.duration)
        sec = detect_secondary_eclipse(ph2, fl2, tls_r.depth)
        cen = centroid_motion_proxy(tc, fc, tls_r.period, tls_r.T0, tls_r.duration)
        oot = out_of_transit_variability(tc, fc, tls_r.period, tls_r.T0, tls_r.duration)

        fv  = compile_feature_vector(tf, oe, sec, cen, oot, tls_r)
        fv['bls_period_days'] = bp
        fv['bls_power']       = round(bpow, 5)
        fv['bls_depth_ppm']   = round(bd * 1e6, 1)
        fv['bls_duration_hours'] = round(bdur * 24, 3)
        fv['star_id']         = star_id
        fv['pipeline_error']  = ''
        return fv

    except Exception as e:
        return {'star_id': star_id, 'pipeline_error': str(e)}


# ─── Example: process multiple stars ───
# Uncomment and set your list of HDF5 files:

# hdf5_files = [
#     'star_001.h5', 'star_002.h5', 'star_003.h5'
# ]
#
# all_features = []
# for f in hdf5_files:
#     t, fl, sid = load_from_hdf5(f)
#     print(f'Processing {sid}...')
#     fv = run_phase2_pipeline(t, fl, sid)
#     all_features.append(fv)
#
# df_batch = pd.DataFrame(all_features)
# df_batch.to_csv('role2_batch_features.csv', index=False)
# print(f'Batch complete: {len(df_batch)} stars processed')
# display(df_batch)

print('Batch processing template ready.')
print('Uncomment the code above and set your list of HDF5 files from Role 1.')

---
## ✅ Cell 15: Phase 2 Summary & Deliverables Checklist

In [ ]:
print('='*60)
print('  PHASE 2 SUMMARY — Feature Engineer & Signal Analyst')
print('='*60)
print(f'  Star: {STAR_ID}')
print(f'  Data points: {len(time_clean)}')
print()
print('  ┌─ PERIOD SEARCH ──────────────────────────────────────┐')
print(f'  │  BLS Best Period  : {best_period_bls:.5f} days              │')
print(f'  │  TLS Best Period  : {tls_result.period:.5f} days              │')
print(f'  │  SDE              : {tls_result.SDE:.2f}                      │')
print(f'  │  FAP              : {tls_result.FAP:.2e}                  │')
print('  ├─ TRANSIT SHAPE ──────────────────────────────────────┤')
print(f'  │  Depth            : {tls_result.depth*1e6:.0f} ppm                     │')
print(f'  │  Duration         : {tls_result.duration*24:.2f} hours                  │')
print(f'  │  Rp/Rs            : {tls_result.rp_rs:.4f}                    │')
print(f'  │  Impact param     : {transit_features["impact_param_proxy"]:.4f}                    │')
print('  ├─ VETTING FLAGS ──────────────────────────────────────┤')
print(f'  │  Odd-Even flag    : {oe_result.get("is_suspicious_eb", False)}               │')
print(f'  │  Secondary ecl.   : {sec_result.get("is_suspicious_secondary", False)}               │')
print(f'  │  Contamination    : {centroid_result.get("contamination_flag", False)}               │')
print(f'  │  High variability : {oot_result.get("high_variability_flag", False)}               │')
print(f'  │  Total FP flags   : {feature_vector["n_fp_flags"]}                       │')
print('  ├─ FINAL VERDICT ──────────────────────────────────────┤')
verdict = '🟢 PLANET CANDIDATE' if feature_vector['is_planet_candidate'] else '🔴 FALSE POSITIVE'
print(f'  │  Result: {verdict:46}│')
print('  └──────────────────────────────────────────────────────┘')
print()
print('  📁 DELIVERABLES CHECKLIST:')
deliverables = [
    ('01_raw_light_curve.png',    'Raw light curve plot'),
    ('02_bls_periodogram.png',    'BLS periodogram + phase-folded'),
    ('03_tls_power_spectrum.png', 'TLS power spectrum + phase-folded'),
    (f'04_phase_folded_{STAR_ID}.png', 'Phase-folded thumbnail for ML training'),
    ('05_secondary_eclipse_check.png', 'Secondary eclipse diagnostic plot'),
    (f'role2_feature_table_{STAR_ID}.csv', 'Feature vector CSV for Role 3 GBM'),
]
for fname, desc in deliverables:
    exists = '✅' if os.path.exists(fname) else '⏳'
    print(f'  {exists}  {fname}')
    print(f'     → {desc}')
print()
print('  🤝 HANDOFF:')
print('  → CSV feature table goes to Role 3 (ML modeller)')
print('  → PNG thumbnails go to Role 3 (1D-CNN training)')
print('  → This notebook goes to GitHub: feature/role2-signal-analysis')